# V2a-RSNs Clustered IC Selection from Saved Linear Decompositions

This notebook starts from the decomposition artifacts saved by `linear_methods.ipynb` and only performs spectral clustering, cluster-level IC selection, and trace reconstruction. Run `linear_methods.ipynb` first with `SAVE_DECOMPOSITION_OUTPUTS = True`; this notebook then reads `components`, `spectra`, `mixing`, `mean`, and `metadata` from `outputs/linear/<dataset_group>/<data_name>/<method>/`.

Clustered cleaned traces are saved under `outputs/linear/<dataset_group>/<data_name>/<method>/cleaned/`, and cluster-selection JSON is saved under `outputs/linear/<dataset_group>/<data_name>/<method>/clusters/` when `SAVE_OUTPUTS = True`.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    p
    for p in candidates
    if (p / "pyproject.toml").exists() and (p / "src" / "bss_notebook.py").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
CORE_DIR = SRC_DIR / "core"
for path in (SRC_DIR, CORE_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from bss_notebook import (
    BSS_METHODS,
    available_datasets,
    load_bss_decomposition_outputs,
    load_traces,
    output_directory,
    save_cleaned_trace_output,
    save_cluster_selection_output,
    summarize_decomposition_results,
)
from ica_utils import (
    cluster,
    rank_clusters_by_mean_log_psd,
    reconstruct_bss,
    reject_components_from_cluster_selection,
)

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PROJECT_ROOT


In [ ]:
available_datasets(PROJECT_ROOT)


In [ ]:
# Dataset and method selection
DATASET_KEY = "v2a-RSNs/220127_F4_run2_fluorescence"

# Use list(BSS_METHODS) to load all four saved linear decompositions, or choose a subset.
METHODS_TO_RUN = list(BSS_METHODS) # ["fastica", "infomax"] or list(BSS_METHODS)

# Read decomposition artifacts produced by notebooks/v2a-RSNs/linear_methods.ipynb.
SOURCE_ANALYSIS_KIND = "linear"

# Clustered output layout:
#   outputs/linear/<dataset_group>/<data_name>/<method>/cleaned/
#   outputs/linear/<dataset_group>/<data_name>/<method>/clusters/
OUTPUT_ANALYSIS_KIND = "linear"

# Cluster ICs by Welch-spectrum features, matching the reference ICA notebook's clustering workflow.
N_CLUSTERS = 7
CLUSTER_FEATURE_START_BIN = 30
CLUSTER_NPERSEG = 250
CLUSTER_NOVERLAP = 125
CLUSTER_RANDOM_STATE = 0

# Set True after cluster rejection choices are final.
SAVE_OUTPUTS = True


In [ ]:
dataset, traces = load_traces(DATASET_KEY, PROJECT_ROOT)
n_neurons, n_frames = traces.shape
sample_rate_hz = dataset.sample_rate_hz or 1.0

print(f"dataset: {dataset.key}")
print(f"trace file: {dataset.trace_path.relative_to(PROJECT_ROOT)}")
print(f"traces: neurons={n_neurons}, frames={n_frames}")
print(f"methods: {METHODS_TO_RUN}")
for method in METHODS_TO_RUN:
    method_dir = output_directory(
        method,
        dataset.data_name,
        PROJECT_ROOT,
        analysis_kind=OUTPUT_ANALYSIS_KIND,
        dataset_group=dataset.group,
    )
    print(f"{method} linear decomposition dir: {method_dir.relative_to(PROJECT_ROOT)}")
    print(f"{method} cleaned output dir: {(method_dir / 'cleaned').relative_to(PROJECT_ROOT)}")
    print(f"{method} cluster metadata dir: {(method_dir / 'clusters').relative_to(PROJECT_ROOT)}")


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
time_slice = slice(0, min(500, n_frames))
shown_neurons = list(range(min(6, n_neurons)))
offset = np.nanstd(traces[:, time_slice]) * 0.4
if not np.isfinite(offset) or offset == 0:
    offset = 1.0

for row_idx, neuron_idx in enumerate(shown_neurons):
    ax.plot(
        np.arange(time_slice.start, time_slice.stop),
        traces[neuron_idx, time_slice] + row_idx * offset,
        lw=0.8,
    )

ax.set_title("Raw traces")
ax.set_xlabel("frame")
ax.set_ylabel("neuron")
ax.set_yticks([row_idx * offset for row_idx in range(len(shown_neurons))])
ax.set_yticklabels([str(neuron_idx) for neuron_idx in shown_neurons]);


In [ ]:
import time

results = {}
source_metadata_by_method = {}
total_methods = len(METHODS_TO_RUN)
load_start = time.perf_counter()

for idx, method in enumerate(METHODS_TO_RUN, start=1):
    t0 = time.perf_counter()
    print(f"[{idx}/{total_methods}] Loading saved linear decomposition artifacts for {method}...")
    result = load_bss_decomposition_outputs(
        dataset_key=DATASET_KEY,
        method=method,
        project_root=PROJECT_ROOT,
        analysis_kind=SOURCE_ANALYSIS_KIND,
    )
    results[method] = result
    metadata_path = result.saved_paths["metadata"]
    source_metadata_by_method[method] = json.loads(metadata_path.read_text(encoding="utf-8"))
    dt = time.perf_counter() - t0
    print(
        f"[{idx}/{total_methods}] Loaded {method} in {dt:.1f}s "
        f"(ICs={result.ic_comps.shape}, mixing={result.A.shape})"
    )

print(f"Loaded saved linear decomposition artifacts for {total_methods} methods in {time.perf_counter() - load_start:.1f}s")
summarize_decomposition_results(results)


In [ ]:
component_count = min(8, max(result.ic_comps.shape[1] for result in results.values()))
fig, axes = plt.subplots(len(results), 1, figsize=(14, 2.0 * len(results)), sharex=True)
if len(results) == 1:
    axes = [axes]
for ax, (method, result) in zip(axes, results.items()):
    for comp_idx in range(min(component_count, result.ic_comps.shape[1])):
        ax.plot(result.ic_comps[:, comp_idx] + comp_idx * 3, lw=0.7, label=f"IC {comp_idx}")
    ax.set_title(f"{method.upper()} first {component_count} loaded components")
axes[-1].set_xlabel("frame");


In [ ]:
import time

cluster_results = {}
cluster_rows = []
total_methods = len(results)
cluster_start = time.perf_counter()

for idx, (method, result) in enumerate(results.items(), start=1):
    t0 = time.perf_counter()
    print(f"[{idx}/{total_methods}] Clustering ICs for {method}...")
    new_mat, predictions, spectra, features = cluster(
        result.ic_comps,
        N_CLUSTERS,
        sample_rate_hz,
        feature_start_bin=CLUSTER_FEATURE_START_BIN,
        nperseg=CLUSTER_NPERSEG,
        noverlap=CLUSTER_NOVERLAP,
        random_state=CLUSTER_RANDOM_STATE,
    )
    cluster_results[method] = {
        "new_mat": new_mat,
        "predictions": predictions,
        "spectra": spectra,
        "features": features,
    }
    for cluster_id in sorted(np.unique(predictions)):
        ic_indices = np.flatnonzero(predictions == cluster_id)
        cluster_rows.append(
            {
                "method": method,
                "cluster": int(cluster_id),
                "n_ics": int(ic_indices.size),
                "ic_indices": ic_indices.tolist(),
            }
        )
    dt = time.perf_counter() - t0
    print(f"[{idx}/{total_methods}] Finished clustering {method} in {dt:.1f}s")

print(f"Completed clustering for {total_methods} methods in {time.perf_counter() - cluster_start:.1f}s")
pd.DataFrame(cluster_rows)


In [ ]:
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

for method, clustered in cluster_results.items():
    print(method)

    new_mat = clustered["new_mat"]
    predictions = clustered["predictions"]
    spectra = clustered["spectra"]

    ranked = rank_clusters_by_mean_log_psd(
        spectra,
        predictions,
        sample_rate_hz,
        fmin=0.0,
        fmax=0.20,
        aggregate="peak",
    )
    ordered_clusters = [row["cluster"] for row in ranked]
    print(f"  Peak log-PSD rank near 0 Hz (high→low): {ordered_clusters}")

    fig = plt.figure(figsize=(12, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.1, 1.0])

    # Left: 3D cluster scatter
    ax3d = fig.add_subplot(gs[0, 0], projection="3d")
    for label in np.unique(predictions):
        pts = new_mat[predictions == label]
        ax3d.scatter(pts[:, 0], pts[:, 1], pts[:, 2], label=f"clus {int(label)}")
    ax3d.set_xlabel("x-axis")
    ax3d.set_ylabel("y-axis")
    ax3d.set_zlabel("z-axis")
    ax3d.set_title(f"{method}: IC Clusters")
    ax3d.legend(loc="best")
    ax3d.view_init(elev=22, azim=45)

    # Right: overlay of mean log-PSDs only
    ax_psd = fig.add_subplot(gs[0, 1])
    eps = np.finfo(float).eps
    freqs = np.linspace(0.0, float(sample_rate_hz) / 2.0, spectra.shape[1])
    for label in np.unique(predictions):
        group = spectra[predictions == label]
        mean_log = np.log(np.maximum(group.mean(axis=0), eps))
        ax_psd.plot(freqs, mean_log, lw=1.5, label=f"clus {int(label)}")

    ax_psd.set_xlim([0, 0.75])
    ax_psd.set_ylim([-3, 3])
    ax_psd.set_xlabel("Frequency (Hz)")
    ax_psd.set_ylabel("Log PSD")
    ax_psd.set_title(f"{method}: Mean Log-PSD Overlay")
    ax_psd.text(0.01, 0.02, f"peak-rank (0-0.20 Hz): {ordered_clusters}", transform=ax_psd.transAxes, fontsize=9)
    ax_psd.legend(loc="best")

    fig.tight_layout()
    display(fig)
    plt.close(fig)



In [ ]:
# Set True after cluster rejection choices are final.
SAVE_OUTPUTS = True

# Fill after inspecting the 3D cluster plots. If keep_clusters is non-empty, all other clusters are rejected.
CLUSTER_SELECTION_BY_METHOD = {
    "fastica": {"reject_clusters": [], "keep_clusters": [2, 6, 3, 4]},
    "infomax": {"reject_clusters": [], "keep_clusters": [1, 5, 3, 0]},
}

In [ ]:
clustered_cleaned = {}
reject_components_by_method = {}
accepted_components_by_method = {}
accepted_counts_by_method = {}
selection_rows = []

for method, result in results.items():
    selection = CLUSTER_SELECTION_BY_METHOD.get(method, {})
    predictions = cluster_results[method]["predictions"]
    reject_components = reject_components_from_cluster_selection(
        predictions,
        reject_clusters=selection.get("reject_clusters", []),
        keep_clusters=selection.get("keep_clusters", []),
    )
    all_components = np.arange(predictions.size, dtype=int)
    accepted_components = np.setdiff1d(all_components, reject_components)

    reject_components_by_method[method] = reject_components.tolist()
    accepted_components_by_method[method] = accepted_components.tolist()
    accepted_counts_by_method[method] = int(accepted_components.size)

    clustered_cleaned[method] = reconstruct_bss(
        result.ic_comps,
        result.A,
        result.mean,
        reject=reject_components,
    )
    selection_rows.append(
        {
            "method": method,
            "reject_clusters": selection.get("reject_clusters", []),
            "keep_clusters": selection.get("keep_clusters", []),
            "accept_components": accepted_components.tolist(),
            "n_accepted": int(accepted_components.size),
            "reject_components": reject_components.tolist(),
            "n_rejected": int(reject_components.size),
        }
    )

selection_df = pd.DataFrame(selection_rows)
print("Effective n_components per method (from accepted clusters):")
display(selection_df[["method", "n_accepted", "n_rejected", "keep_clusters", "reject_clusters"]])
selection_df



In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(14, 2.3 * len(results)), sharex=True)
if len(results) == 1:
    axes = [axes]
neuron_idx = 0
raw_vs_cleaned_offset = 2
for ax, (method, result) in zip(axes, results.items()):
    raw_trace = traces[neuron_idx, time_slice]
    cleaned_trace = clustered_cleaned[method][time_slice, neuron_idx]
    ax.plot(raw_trace, lw=0.8, label="raw", color="red", alpha=0.65)
    ax.plot(
        cleaned_trace + raw_vs_cleaned_offset,
        lw=0.8,
        label=f"{method} cluster-cleaned (+{raw_vs_cleaned_offset:g} offset)",
        color="blue",
        alpha=0.8,
    )
    ax.set_title(f"Raw vs cluster-cleaned, neuron {neuron_idx}, {method}")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("frame");


In [ ]:
saved_paths_by_method = {}


def _cluster_members_for_method(method):
    predictions = cluster_results[method]["predictions"]
    spectra = cluster_results[method]["spectra"]
    ranked = rank_clusters_by_mean_log_psd(
        spectra,
        predictions,
        sample_rate_hz,
        fmin=0.0,
        fmax=0.20,
        aggregate="peak",
    )
    rank_by_cluster = {int(row["cluster"]): int(row["rank"]) for row in ranked}
    accepted = set(accepted_components_by_method[method])
    members = []
    for cluster_id in sorted(np.unique(predictions)):
        ic_indices = [int(idx) for idx in np.flatnonzero(predictions == cluster_id)]
        kept = [idx for idx in ic_indices if idx in accepted]
        rejected = [idx for idx in ic_indices if idx not in accepted]
        members.append(
            {
                "cluster": int(cluster_id),
                "n_ics": len(ic_indices),
                "ic_indices": ic_indices,
                "accepted_ic_indices": kept,
                "rejected_ic_indices": rejected,
                "peak_log_psd_rank_0_to_0p20_hz": rank_by_cluster.get(int(cluster_id)),
            }
        )
    return members


if SAVE_OUTPUTS:
    for method, result in results.items():
        method_dir = output_directory(
            method,
            result.dataset.data_name,
            PROJECT_ROOT,
            analysis_kind=OUTPUT_ANALYSIS_KIND,
            dataset_group=result.dataset.group,
        )
        cleaned_paths = save_cleaned_trace_output(
            spec=result.dataset,
            method=method,
            traces=result.traces,
            cleaned=clustered_cleaned[method],
            reject_components=reject_components_by_method[method],
            output_dir=method_dir,
            metadata={
                "source_decomposition_dir": str(result.output_dir.relative_to(PROJECT_ROOT)),
                "selection_strategy": "spectral_cluster_selection",
                "cluster_parameters": {
                    "n_clusters": int(N_CLUSTERS),
                    "feature_start_bin": int(CLUSTER_FEATURE_START_BIN),
                    "nperseg": int(CLUSTER_NPERSEG),
                    "noverlap": int(CLUSTER_NOVERLAP),
                    "random_state": int(CLUSTER_RANDOM_STATE),
                },
                "accepted_components": accepted_components_by_method[method],
                "rejected_components": reject_components_by_method[method],
            },
        )
        selection = CLUSTER_SELECTION_BY_METHOD.get(method, {})
        selection_path = save_cluster_selection_output(
            spec=result.dataset,
            method=method,
            output_dir=method_dir,
            selection={
                "source_decomposition_dir": str(result.output_dir.relative_to(PROJECT_ROOT)),
                "source_decomposition_metadata": source_metadata_by_method.get(method, {}),
                "cleaned_trace_path": str(cleaned_paths["cleaned"].relative_to(PROJECT_ROOT)),
                "cleaned_metadata_path": str(cleaned_paths["cleaned_metadata"].relative_to(PROJECT_ROOT)),
                "cluster_parameters": {
                    "n_clusters": int(N_CLUSTERS),
                    "feature_start_bin": int(CLUSTER_FEATURE_START_BIN),
                    "nperseg": int(CLUSTER_NPERSEG),
                    "noverlap": int(CLUSTER_NOVERLAP),
                    "random_state": int(CLUSTER_RANDOM_STATE),
                },
                "requested_selection": {
                    "keep_clusters": [int(value) for value in selection.get("keep_clusters", [])],
                    "reject_clusters": [int(value) for value in selection.get("reject_clusters", [])],
                },
                "accepted_components": accepted_components_by_method[method],
                "rejected_components": reject_components_by_method[method],
                "n_accepted_components": accepted_counts_by_method[method],
                "n_rejected_components": len(reject_components_by_method[method]),
                "clusters": _cluster_members_for_method(method),
            },
        )
        saved_paths_by_method[method] = {
            "cleaned": cleaned_paths["cleaned"],
            "cleaned_metadata": cleaned_paths["cleaned_metadata"],
            "cluster_selection": selection_path,
        }

for method, result in results.items():
    accepted_count = accepted_counts_by_method.get(method)
    method_dir = output_directory(
        method,
        result.dataset.data_name,
        PROJECT_ROOT,
        analysis_kind=OUTPUT_ANALYSIS_KIND,
        dataset_group=result.dataset.group,
    )
    if accepted_count is not None:
        print(f"{method}: accepted components count = {accepted_count}")
    if method in saved_paths_by_method:
        print(f"{method}:")
        for label, path in saved_paths_by_method[method].items():
            print(f"  {label}: {path.relative_to(PROJECT_ROOT)}")
    else:
        print(
            f"{method}: not saved; set SAVE_OUTPUTS = True after cluster selections are final "
            f"to write cleaned traces under {(method_dir / 'cleaned').relative_to(PROJECT_ROOT)}"
        )
